# Cross-Validation and Model Selection

Every other walkthrough in this section answers *how does this algorithm work*.
This one answers a different question: **how do you know whether it works**, and
why the obvious way of checking is wrong often enough to matter.

The short version: a single train/test split gives you one number, and that
number has a standard error nobody prints. Cross-validation gives you a
distribution instead. And the moment you start *choosing* between models using
that distribution, you introduce a new bias that needs a second layer of
validation to remove.

Everything here is worked on 12 EEG recordings so the folds can be written out
by hand, then checked against scikit-learn.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True, linewidth=140)


def banner(title, ch="="):
    print(ch * 78)
    print(f"  {title}")
    print(ch * 78)


def step(label):
    print()
    print(label)
    print("=" * 78)


def check(manual, from_array, label="MATCH"):
    ok = np.allclose(manual, from_array)
    print(f"      = {manual:+.6f}   |  from array: {from_array:+.6f}   |  {label}: {ok}")
    return ok


def expand(u, v, per_line=4):
    """Write a dot product out term by term."""
    terms = [f"({a:+.4f})*({b:+.4f})" for a, b in zip(u, v)]
    lines, out = [], ""
    for i, t in enumerate(terms):
        out += t + (" + " if i < len(terms) - 1 else "")
        if (i + 1) % per_line == 0:
            lines.append(out); out = ""
    if out:
        lines.append(out)
    return "\n      ".join(lines)


def show_matrix(M, name="M", rows=None, cols=None, fmt="{:+8.3f}", max_cols=14):
    M = np.asarray(M)
    if M.ndim == 1:
        M = M[None, :]
    cut = M.shape[1] > max_cols
    Ms = M[:, :max_cols] if cut else M
    print(f"{name} {np.asarray(M).shape}" +
          (f"   [first {max_cols} of {M.shape[1]} columns]" if cut else "") + " =")
    if cols is not None:
        head = "".join(f"{str(c):>10}" for c in (cols[:max_cols] if cut else cols))
        print(" " * 13 + head)
    for r in range(Ms.shape[0]):
        lab = f"{str(rows[r]):>10} | " if rows is not None else f"{r:>10} | "
        print("   " + lab + "  ".join(fmt.format(v) for v in Ms[r]))


def show_matmul(A, B, C, na="A", nb="B", nc="C", i=0, j=0,
                rows_A=None, note=None, max_cols=14):
    """Print both operands of A @ B = C, then verify one element by hand."""
    A, B, C = np.asarray(A), np.asarray(B), np.asarray(C)
    banner(f"{na} {A.shape}   @   {nb} {B.shape}   =   {nc} {C.shape}")
    print(f"  shape rule: ({A.shape[0]} x {A.shape[1]}) @ ({B.shape[0]} x {B.shape[1]})"
          f" -> ({A.shape[0]} x {B.shape[1]})   inner dim {A.shape[1]} cancels")
    if note:
        print(f"  {note}")
    print("=" * 78)
    print()
    show_matrix(A, na, rows_A, max_cols=max_cols)
    print()
    show_matrix(B, nb, max_cols=max_cols)
    print()
    show_matrix(C, nc, rows_A, max_cols=max_cols)
    print(f"\n  worked element  {nc}[{i},{j}]  =  row {i} of {na}  .  column {j} of {nb}")
    print(f"      row {i} of {na} = {np.round(A[i], 4)}")
    print(f"      col {j} of {nb} = {np.round(B[:, j], 4)}")
    print(f"      {expand(A[i], B[:, j])}")
    check(float(A[i] @ B[:, j]), float(C[i, j]))


def show_elementwise(A, B, C, na, nb, nc, op="+", rows=None):
    A, B, C = np.asarray(A), np.asarray(B), np.asarray(C)
    banner(f"{nc} = {na} {op} {nb}      all three are {A.shape} - element-wise")
    for r in range(A.shape[0]):
        lab = f"{rows[r]:>10}" if rows is not None else f"{r:>10}"
        print(f"\n{lab} |")
        print("   " + f"{na:>14}: " + "  ".join(f"{v:+8.3f}" for v in A[r]))
        print("   " + f"{nb:>14}: " + "  ".join(f"{v:+8.3f}" for v in B[r]))
        print("   " + f"{nc:>14}: " + "  ".join(f"{v:+8.3f}" for v in C[r]))
    print(f"\n  check [0,0]: {A[0,0]:+.6f} {op} {B[0,0]:+.6f} = {C[0,0]:+.6f}   "
          f"MATCH: {np.isclose(A[0,0] + B[0,0] if op == '+' else A[0,0] - B[0,0], C[0,0])}")


print("Display helpers ready: banner, step, check, expand, show_matrix, "
      "show_matmul, show_elementwise.")

# ----------------------------------------------------------------------------
#  A deliberately tiny EEG-flavoured table: 12 recordings, hand-checkable.
# ----------------------------------------------------------------------------
feature_names = ["line_length", "spike_rate", "age"]
region_names  = ["frontal", "temporal", "occipital"]

X = np.array([
    [12.0,  0.5, 24.0],
    [15.0,  1.2, 31.0],
    [42.0,  6.8, 58.0],
    [38.0,  5.1, 47.0],
    [11.0,  0.3, 19.0],
    [55.0,  8.2, 63.0],
    [40.0,  1.9, 35.0],
    [47.0,  7.4, 52.0],
    [14.0,  0.8, 28.0],
    [51.0,  6.1, 60.0],
    [45.0,  2.4, 41.0],
    [35.0,  4.7, 30.0],
])

# categorical column, used by the CatBoost notebook
region = np.array([0, 1, 1, 2, 0, 1, 2, 1, 0, 2, 0, 1])

# binary target: did a seizure follow?
y_class = np.array([0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1])

# continuous target: seizure burden in minutes/hour
y_reg = np.array([0.2, 0.5, 6.4, 4.8, 0.1, 9.1, 3.5, 7.2, 0.3, 6.9, 1.6, 4.1])

n_samples, n_features = X.shape
rec_ids = [f"rec{i:02d}" for i in range(n_samples)]

print(f"{n_samples} recordings x {n_features} features\n")
hdr = f"{'id':>6}  {'region':>10}  " + "  ".join(f"{f:>12}" for f in feature_names)
hdr += f"  {'y_class':>8}  {'y_reg':>7}"
print(hdr)
print("-" * len(hdr))
for i in range(n_samples):
    print(f"{rec_ids[i]:>6}  {region_names[region[i]]:>10}  " +
          "  ".join(f"{v:>12.1f}" for v in X[i]) +
          f"  {y_class[i]:>8}  {y_reg[i]:>7.1f}")
print("-" * len(hdr))
print(f"\nclass balance: {int((y_class==0).sum())} negative, {int((y_class==1).sum())} positive")
print(f"y_reg mean = {y_reg.mean():.4f}, std = {y_reg.std():.4f}")

Display helpers ready: banner, step, check, expand, show_matrix, show_matmul, show_elementwise.
12 recordings x 3 features

    id      region   line_length    spike_rate           age   y_class    y_reg
-------------------------------------------------------------------------------
 rec00     frontal          12.0           0.5          24.0         0      0.2
 rec01    temporal          15.0           1.2          31.0         0      0.5
 rec02    temporal          42.0           6.8          58.0         1      6.4
 rec03   occipital          38.0           5.1          47.0         1      4.8
 rec04     frontal          11.0           0.3          19.0         0      0.1
 rec05    temporal          55.0           8.2          63.0         1      9.1
 rec06   occipital          40.0           1.9          35.0         1      3.5
 rec07    temporal          47.0           7.4          52.0         1      7.2
 rec08     frontal          14.0           0.8          28.0         0      

## 1. The problem: one split gives one number, and the number moves

Hold out 25% of the data, score the model, report it. The trouble is that
"25% of the data" is not one thing — it is a choice, and different choices give
different answers.

Below, the *same model* on the *same data*, with nothing changed but the random
seed that decides who lands in the test set.

In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

banner("THE SAME MODEL, THE SAME DATA, DIFFERENT SPLITS")

step("Ten random 75/25 splits")
print(f"  {'seed':>5}  {'test recordings':>28}  {'accuracy':>9}")
print("  " + "-" * 48)
accs = []
for seed in range(10):
    tr, te = train_test_split(np.arange(n_samples), test_size=3,
                              random_state=seed, stratify=y_class)
    m = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X[tr], y_class[tr])
    a = m.score(X[te], y_class[te])
    accs.append(a)
    print(f"  {seed:>5}  {', '.join(rec_ids[i] for i in te):>28}  {a:>9.3f}")
print("  " + "-" * 48)
accs = np.array(accs)
print(f"  {'mean':>5}  {'':>28}  {accs.mean():>9.3f}")
print(f"  {'range':>5}  {'':>28}  {accs.min():.3f} - {accs.max():.3f}")
print()
print(f"  Ten defensible answers between {accs.min():.0%} and {accs.max():.0%}.")
print("  Reporting any single one of them is reporting a coin flip as a fact.")

step("Worse: the split can change WHICH MODEL WINS")
print(f"  {'seed':>5}  {'tree':>8}  {'logistic':>10}  winner")
print("  " + "-" * 40)
flips = {'tree': 0, 'logistic': 0, 'tie': 0}
for seed in range(10):
    tr, te = train_test_split(np.arange(n_samples), test_size=4,
                              random_state=seed, stratify=y_class)
    t = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X[tr], y_class[tr]).score(X[te], y_class[te])
    l = LogisticRegression(max_iter=2000).fit(X[tr], y_class[tr]).score(X[te], y_class[te])
    w = 'tree' if t > l else ('logistic' if l > t else 'tie')
    flips[w] += 1
    print(f"  {seed:>5}  {t:>8.3f}  {l:>10.3f}  {w}")
print("  " + "-" * 40)
print(f"  tree wins {flips['tree']}x, logistic wins {flips['logistic']}x,"
      f" tied {flips['tie']}x")
print()
print("  If you had run seed 0 and stopped, you would have shipped one model.")
print("  Seed 1 would have shipped the other. Nothing about the models changed.")

  THE SAME MODEL, THE SAME DATA, DIFFERENT SPLITS

Ten random 75/25 splits
   seed               test recordings   accuracy
  ------------------------------------------------
      0           rec10, rec11, rec06      0.667
      1           rec06, rec08, rec03      0.667
      2           rec00, rec11, rec06      0.667
      3           rec11, rec04, rec02      1.000
      4           rec11, rec04, rec02      1.000
      5           rec11, rec02, rec08      1.000
      6           rec09, rec05, rec04      1.000
      7           rec07, rec10, rec06      0.667
      8           rec09, rec05, rec08      1.000
      9           rec06, rec02, rec10      0.667
  ------------------------------------------------
   mean                                    0.833
  range                                0.667 - 1.000

  Ten defensible answers between 67% and 100%.
  Reporting any single one of them is reporting a coin flip as a fact.

Worse: the split can change WHICH MODEL WINS
   seed      tree

## 2. k-fold, by hand

The fix is to stop throwing data away. Split the data into $k$ equal **folds**;
train $k$ times, each time holding out a different fold; every observation is
tested on exactly once, and every observation trains the model $k-1$ times.

$$\text{CV}_k = \frac{1}{k}\sum_{i=1}^{k} \text{score}\big(f^{(-i)},\ \text{fold}_i\big)$$

where $f^{(-i)}$ is the model trained on everything except fold $i$.

With 12 recordings and $k = 4$, each fold is exactly 3 recordings — small enough
to write every fold out and check the arithmetic.

In [3]:
banner("4-FOLD CROSS-VALIDATION, WRITTEN OUT")

k = 4
idx = np.arange(n_samples)
folds = [idx[i::k] for i in range(k)]     # simple interleaved assignment

step("The folds")
for i, f in enumerate(folds):
    print(f"  fold {i}: {[rec_ids[j] for j in f]}"
          f"   classes {y_class[f].tolist()}")

step("Train k times, hold out one fold each time")
print(f"  {'fold':>5}  {'train on':>9}  {'test on':>26}  {'correct':>9}  {'acc':>7}")
print("  " + "-" * 64)
scores, n_correct = [], 0
for i, f in enumerate(folds):
    tr = np.concatenate([folds[j] for j in range(k) if j != i])
    m  = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X[tr], y_class[tr])
    pred = m.predict(X[f])
    ok = int((pred == y_class[f]).sum())
    n_correct += ok
    s = ok / len(f)
    scores.append(s)
    print(f"  {i:>5}  {len(tr):>9}  {', '.join(rec_ids[j] for j in f):>26}"
          f"  {ok:>4}/{len(f):<4}  {s:>7.3f}")
print("  " + "-" * 64)
scores = np.array(scores)
print(f"  CV_4 = ({' + '.join(f'{s:.3f}' for s in scores)}) / {k}"
      f" = {scores.mean():.4f}")
print()
print(f"  Every recording was tested exactly once:"
      f" {n_correct}/{n_samples} correct overall = {n_correct/n_samples:.4f}")
print(f"  (equal to the fold mean here only because the folds are the same size)")

step("Check against scikit-learn")
from sklearn.model_selection import cross_val_score, KFold
cv = KFold(n_splits=k, shuffle=False)
sk = cross_val_score(DecisionTreeClassifier(max_depth=2, random_state=0),
                     X, y_class, cv=list((np.setdiff1d(idx, f), f) for f in folds))
print(f"  by hand      : {np.round(scores, 4)}   mean {scores.mean():.4f}")
print(f"  scikit-learn : {np.round(sk, 4)}   mean {sk.mean():.4f}")
print(f"  max|diff| = {np.abs(scores-sk).max():.2e}   MATCH: "
      f"{np.allclose(scores, sk)}")

  4-FOLD CROSS-VALIDATION, WRITTEN OUT

The folds
  fold 0: ['rec00', 'rec04', 'rec08']   classes [0, 0, 0]
  fold 1: ['rec01', 'rec05', 'rec09']   classes [0, 1, 1]
  fold 2: ['rec02', 'rec06', 'rec10']   classes [1, 1, 0]
  fold 3: ['rec03', 'rec07', 'rec11']   classes [1, 1, 1]

Train k times, hold out one fold each time
   fold   train on                     test on    correct      acc
  ----------------------------------------------------------------
      0          9         rec00, rec04, rec08     3/3       1.000
      1          9         rec01, rec05, rec09     3/3       1.000
      2          9         rec02, rec06, rec10     2/3       0.667
      3          9         rec03, rec07, rec11     1/3       0.333
  ----------------------------------------------------------------
  CV_4 = (1.000 + 1.000 + 0.667 + 0.333) / 4 = 0.7500

  Every recording was tested exactly once: 9/12 correct overall = 0.7500
  (equal to the fold mean here only because the folds are the same size)

Che

## 3. The number nobody prints: the standard error

A CV score is a **mean of $k$ numbers**, so like any mean it has a spread. The
usual estimate is

$$\text{SE} = \frac{s}{\sqrt{k}}, \qquad s = \text{standard deviation of the fold scores}$$

Reporting `CV = 0.83` without it invites a comparison that the data cannot
support. Two models differing by 0.02 with a standard error of 0.09 are not
distinguishable — and this is the single most common way model comparisons go
wrong.

A caution the formula hides: the $k$ fold scores are **not independent**, because
their training sets overlap heavily (any two share $k-2$ folds). So $s/\sqrt{k}$
*understates* the true uncertainty. It is a floor, not an estimate — treat a
difference smaller than one SE as no difference at all.

In [4]:
banner("HOW UNCERTAIN IS A CV SCORE?")

step("Fold-to-fold spread on our 12 recordings")
s = scores.std(ddof=1)
se = s / np.sqrt(k)
print(f"  fold scores  : {np.round(scores, 4)}")
print(f"  mean         : {scores.mean():.4f}")
print(f"  std (ddof=1) : {s:.4f}")
print(f"  SE = s/sqrt(k) = {s:.4f}/sqrt({k}) = {se:.4f}")
print()
print(f"  report: {scores.mean():.3f} +/- {se:.3f}")

step("Why that matters when comparing models")
from sklearn.naive_bayes import GaussianNB
cands = {
    'tree depth 1'  : DecisionTreeClassifier(max_depth=1, random_state=0),
    'tree depth 2'  : DecisionTreeClassifier(max_depth=2, random_state=0),
    'tree depth 4'  : DecisionTreeClassifier(max_depth=4, random_state=0),
    'logistic'      : LogisticRegression(max_iter=2000),
    'naive bayes'   : GaussianNB(),
}
splits = [(np.setdiff1d(idx, f), f) for f in folds]
print(f"  {'model':>14}  {'CV mean':>8}  {'SE':>7}  fold scores")
print("  " + "-" * 60)
res = {}
for name, est in cands.items():
    sc = cross_val_score(est, X, y_class, cv=splits)
    res[name] = sc
    print(f"  {name:>14}  {sc.mean():>8.4f}  {sc.std(ddof=1)/np.sqrt(k):>7.4f}"
          f"  {np.round(sc, 3)}")
print("  " + "-" * 60)

best = max(res, key=lambda n: res[n].mean())
bm, bse = res[best].mean(), res[best].std(ddof=1)/np.sqrt(k)
print(f"  best mean: '{best}' at {bm:.4f} (SE {bse:.4f})")
print()
print(f"  Within one SE of the best -- i.e. NOT distinguishable from it:")
for name in res:
    if res[name].mean() >= bm - bse:
        print(f"    {name:>14}  {res[name].mean():.4f}")
print()
print("  With 12 recordings the data simply cannot separate these. Picking the")
print("  top row and claiming it is 'the best model' overstates what you know.")

step("The 1-SE rule")
print("  A standard response: among models within one SE of the best, choose")
print("  the SIMPLEST rather than the highest-scoring. It costs nothing in")
print("  measurable accuracy and buys robustness, because the ranking inside")
print("  that band is mostly noise.")

  HOW UNCERTAIN IS A CV SCORE?

Fold-to-fold spread on our 12 recordings
  fold scores  : [1.     1.     0.6667 0.3333]
  mean         : 0.7500
  std (ddof=1) : 0.3191
  SE = s/sqrt(k) = 0.3191/sqrt(4) = 0.1596

  report: 0.750 +/- 0.160

Why that matters when comparing models
           model   CV mean       SE  fold scores
  ------------------------------------------------------------
    tree depth 1    0.9167   0.0833  [1.    1.    0.667 1.   ]
    tree depth 2    0.7500   0.1596  [1.    1.    0.667 0.333]
    tree depth 4    0.9167   0.0833  [1.    1.    0.667 1.   ]
        logistic    0.9167   0.0833  [1.    1.    0.667 1.   ]
     naive bayes    0.9167   0.0833  [1.    1.    0.667 1.   ]
  ------------------------------------------------------------
  best mean: 'tree depth 1' at 0.9167 (SE 0.0833)

  Within one SE of the best -- i.e. NOT distinguishable from it:
      tree depth 1  0.9167
      tree depth 4  0.9167
          logistic  0.9167
       naive bayes  0.9167

  With 

## 4. Choosing the right splitter

Plain k-fold assumes every row is an independent draw from the same
distribution. When that assumption fails — and it usually does — plain k-fold
gives an answer that is confidently wrong. Each variant below fixes one specific
violation.

| Splitter | Use when | What breaks without it |
|---|---|---|
| **Stratified** k-fold | classification, especially imbalanced | a fold with no positives makes some metrics undefined and all of them unstable |
| **Leave-one-out** | very small $n$ | nothing breaks, but the estimate has high variance and costs $n$ fits |
| **Group** k-fold | repeated measures — several rows per patient/subject/session | the same subject appears in train and test: you measure memorisation, not generalisation |
| **TimeSeriesSplit** | anything ordered in time | training on the future to predict the past |

### 4.1 Stratified

Plain k-fold on 12 recordings with 7 positives can easily produce a fold that is
all one class. Stratified k-fold keeps each fold's class ratio close to the
whole dataset's.

In [5]:
banner("STRATIFIED K-FOLD")

from sklearn.model_selection import StratifiedKFold, LeaveOneOut, GroupKFold, TimeSeriesSplit

step("Plain k-fold: what the class balance does")
print(f"  overall: {int((y_class==0).sum())} negative,"
      f" {int((y_class==1).sum())} positive"
      f"  ({y_class.mean():.0%} positive)")
print()
print(f"  {'fold':>5}  {'test classes':>16}  {'% positive':>11}")
print("  " + "-" * 38)
kf = KFold(n_splits=4, shuffle=True, random_state=3)
for i, (_, te) in enumerate(kf.split(X)):
    print(f"  {i:>5}  {str(y_class[te].tolist()):>16}"
          f"  {y_class[te].mean():>11.0%}")
print("  " + "-" * 38)

step("Stratified: each fold mirrors the whole")
print(f"  {'fold':>5}  {'test classes':>16}  {'% positive':>11}")
print("  " + "-" * 38)
skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=3)
ratios = []
for i, (_, te) in enumerate(skf.split(X, y_class)):
    ratios.append(y_class[te].mean())
    print(f"  {i:>5}  {str(y_class[te].tolist()):>16}"
          f"  {y_class[te].mean():>11.0%}")
print("  " + "-" * 38)
print(f"  spread of fold positive-rate: plain vs stratified")
plain = [y_class[te].mean() for _, te in KFold(4, shuffle=True, random_state=3).split(X)]
print(f"    plain      : {np.round(plain, 3).tolist()}"
      f"   range {max(plain)-min(plain):.3f}")
print(f"    stratified : {np.round(ratios, 3).tolist()}"
      f"   range {max(ratios)-min(ratios):.3f}")
print()
print("  With rarer positives this stops being cosmetic: a fold with zero")
print("  positives makes recall undefined and AUC uncomputable.")

step("Leave-one-out: k = n")
loo_scores = []
for tr, te in LeaveOneOut().split(X):
    m = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X[tr], y_class[tr])
    loo_scores.append(int(m.predict(X[te])[0] == y_class[te][0]))
loo_scores = np.array(loo_scores)
print(f"  {n_samples} fits, each testing one recording")
print(f"  per-recording correct: {loo_scores.tolist()}")
print(f"  LOO accuracy = {loo_scores.sum()}/{n_samples} = {loo_scores.mean():.4f}")
print()
print(f"  compare 4-fold: {scores.mean():.4f}")
print("  LOO uses the most training data (n-1 rows) so it is nearly unbiased,")
print("  but each test set is a single row, so the estimate is high-variance")
print("  and it costs n fits. For n=12 that is fine; for n=100k it is not.")

  STRATIFIED K-FOLD

Plain k-fold: what the class balance does
  overall: 5 negative, 7 positive  (58% positive)

   fold      test classes   % positive
  --------------------------------------
      0         [0, 0, 1]          33%
      1         [1, 1, 1]         100%
      2         [0, 1, 1]          67%
      3         [0, 1, 0]          33%
  --------------------------------------

Stratified: each fold mirrors the whole
   fold      test classes   % positive
  --------------------------------------
      0         [0, 0, 1]          33%
      1         [1, 1, 0]          67%
      2         [0, 1, 1]          67%
      3         [0, 1, 1]          67%
  --------------------------------------
  spread of fold positive-rate: plain vs stratified
    plain      : [0.333, 1.0, 0.667, 0.333]   range 0.667
    stratified : [0.333, 0.667, 0.667, 0.667]   range 0.333

  With rarer positives this stops being cosmetic: a fold with zero
  positives makes recall undefined and AUC uncomputab

### 4.2 Group k-fold — the one that bites EEG work

Suppose the 12 recordings come from 6 patients, two recordings each. Two
recordings from the same patient are *not* independent: they share anatomy,
electrode placement, baseline rhythms.

Plain k-fold will happily put patient 3's first recording in the training set and
their second in the test set. The model can then score well by recognising the
patient rather than the pathology — and the reported accuracy is measuring
something you will never have at deployment, when every patient is new.

Group k-fold keeps all rows from a group on the same side of every split.

In [6]:
banner("GROUP K-FOLD: WHEN ROWS ARE NOT INDEPENDENT")

# 6 patients, 2 recordings each
patient = np.array([0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5])

step("The data has hidden structure")
print(f"  {'id':>6}  {'patient':>8}  {'y':>3}")
for i in range(n_samples):
    print(f"  {rec_ids[i]:>6}  {'P'+str(patient[i]):>8}  {y_class[i]:>3}")

step("Plain k-fold splits patients across train and test")
pats = lambda a: sorted(int(v) for v in set(a))     # plain ints, not np.int64
kf = KFold(n_splits=3, shuffle=True, random_state=1)
leaks = 0
for i, (tr, te) in enumerate(kf.split(X)):
    both = pats(set(patient[tr]) & set(patient[te]))
    leaks += len(both)
    print(f"  fold {i}: test patients {pats(patient[te])}"
          f"  |  ALSO IN TRAIN: {both}")
print(f"\n  {leaks} patient-level leaks across 3 folds.")

step("GroupKFold keeps each patient whole")
gkf = GroupKFold(n_splits=3)
for i, (tr, te) in enumerate(gkf.split(X, y_class, groups=patient)):
    both = pats(set(patient[tr]) & set(patient[te]))
    print(f"  fold {i}: test patients {pats(patient[te])}"
          f"  |  also in train: {both if both else 'none'}")

step("What the difference is worth, in accuracy")
def cv_acc(splitter, **kw):
    out = []
    for tr, te in splitter.split(X, y_class, **kw):
        m = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X[tr], y_class[tr])
        out.append(m.score(X[te], y_class[te]))
    return np.array(out)

a_plain = cv_acc(KFold(n_splits=3, shuffle=True, random_state=1))
a_group = cv_acc(GroupKFold(n_splits=3), groups=patient)
print(f"  plain  3-fold : {np.round(a_plain, 3)}  mean {a_plain.mean():.4f}")
print(f"  group  3-fold : {np.round(a_group, 3)}  mean {a_group.mean():.4f}")
print(f"  optimism from ignoring groups:"
      f" {a_plain.mean()-a_group.mean():+.4f}")
print()
print("  The group number is the one that predicts performance on a NEW")
print("  patient, which is the only question anyone actually asked.")

  GROUP K-FOLD: WHEN ROWS ARE NOT INDEPENDENT

The data has hidden structure
      id   patient    y
   rec00        P0    0
   rec01        P0    0
   rec02        P1    1
   rec03        P1    1
   rec04        P2    0
   rec05        P2    1
   rec06        P3    1
   rec07        P3    1
   rec08        P4    0
   rec09        P4    1
   rec10        P5    0
   rec11        P5    1

Plain k-fold splits patients across train and test
  fold 0: test patients [1, 2, 5]  |  ALSO IN TRAIN: [2, 5]
  fold 1: test patients [0, 3]  |  ALSO IN TRAIN: []
  fold 2: test patients [2, 4, 5]  |  ALSO IN TRAIN: [2, 5]

  4 patient-level leaks across 3 folds.

GroupKFold keeps each patient whole
  fold 0: test patients [2, 5]  |  also in train: none
  fold 1: test patients [1, 4]  |  also in train: none
  fold 2: test patients [0, 3]  |  also in train: none

What the difference is worth, in accuracy
  plain  3-fold : [0.75 0.75 1.  ]  mean 0.8333
  group  3-fold : [0.75 1.   0.75]  mean 0.8333
  op

### 4.3 Time series — shuffling is cheating

If rows are ordered in time, a random split trains on the future to predict the
past. The model can learn from information that would not exist at prediction
time, and the score is meaningless.

`TimeSeriesSplit` produces expanding windows: always train on a prefix, always
test on the block immediately after it. Note the consequence — the training set
size differs between folds, so the fold scores are not identically distributed,
and averaging them is a rougher approximation than in plain k-fold.

In [7]:
banner("TIME-SERIES SPLITTING")

step("Expanding-window folds")
tss = TimeSeriesSplit(n_splits=4)
print(f"  {'fold':>5}  {'train (in time order)':>34}  {'test':>14}")
print("  " + "-" * 58)
for i, (tr, te) in enumerate(tss.split(X)):
    print(f"  {i:>5}  {str(tr.tolist()):>34}  {str(te.tolist()):>14}")
print("  " + "-" * 58)
print("  Train indices are always BEFORE test indices. No fold ever sees the")
print("  future. Note the training set grows: fold 0 trains on 4 rows, fold 3")
print("  on 10 -- so the folds are not measuring quite the same thing.")

step("What a random split would do here")
kf = KFold(n_splits=4, shuffle=True, random_state=0)
viol = 0
for i, (tr, te) in enumerate(kf.split(X)):
    bad = [int(t) for t in te if (tr > t).any()]
    viol += len(bad)
    print(f"  fold {i}: test {te.tolist()}"
          f"  -> {len(bad)} test rows precede some training row")
print(f"\n  {viol} causality violations. Every one is the model being handed")
print("  information that will not exist when it runs for real.")

  TIME-SERIES SPLITTING

Expanding-window folds
   fold               train (in time order)            test
  ----------------------------------------------------------
      0                        [0, 1, 2, 3]          [4, 5]
      1                  [0, 1, 2, 3, 4, 5]          [6, 7]
      2            [0, 1, 2, 3, 4, 5, 6, 7]          [8, 9]
      3      [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]        [10, 11]
  ----------------------------------------------------------
  Train indices are always BEFORE test indices. No fold ever sees the
  future. Note the training set grows: fold 0 trains on 4 rows, fold 3
  on 10 -- so the folds are not measuring quite the same thing.

What a random split would do here
  fold 0: test [4, 6, 11]  -> 2 test rows precede some training row
  fold 1: test [2, 8, 10]  -> 3 test rows precede some training row
  fold 2: test [1, 7, 9]  -> 3 test rows precede some training row
  fold 3: test [0, 3, 5]  -> 3 test rows precede some training row

  11 causality viol

## 5. Nested CV — the bias you create by choosing

Here is the subtle one, and the reason this page exists.

Section 3 used cross-validation to *compare* five models and pick the best. But
the winning score was the **maximum** over five noisy estimates — and the maximum
of noisy numbers is biased upward. Some of that model's advantage is real skill,
and some is simply that it got the friendlier folds.

Report that maximum as your expected performance and you are reporting a number
that includes the luck. The more candidates you try, the worse it gets.

**Nested cross-validation** separates the two jobs:

- an **inner** loop searches hyperparameters, using only the training part of each outer fold
- an **outer** loop scores the whole *procedure* — "tune, then fit" — on data the inner loop never saw

The outer score answers the honest question: *if I run my entire model-selection
process on new data, how well will the result do?*

The demonstration below is the important one on this page. It uses pure noise —
features with **no relationship to the target whatsoever**, where the true
accuracy of any model is chance. A flat (non-nested) search still reports well
above chance. Nested CV does not.

In [8]:
banner("SELECTION BIAS: FLAT SEARCH vs NESTED CV, ON PURE NOISE")

from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

rng = np.random.RandomState(0)
n_noise, p_noise = 60, 200
Xn = rng.randn(n_noise, p_noise)                    # random features
yn = np.array([0, 1] * (n_noise // 2))              # EXACTLY balanced
rng.shuffle(yn)                                     # ...and no signal

print(f"  {n_noise} samples, {p_noise} features, all random.")
print(f"  class balance: {np.bincount(yn).tolist()} -- deliberately exact, so")
print(f"  the majority-class baseline is {max(np.bincount(yn))/n_noise:.2f} and"
      f" cannot be mistaken for skill.")
print(f"  There is no relationship to learn. Honest accuracy = 0.50.")

# Candidate models: logistic regression on 40 different random 5-feature
# subsets. Fixed up front, so every fold sees the same candidate family.
n_cand = 40
subsets = [rng.choice(p_noise, 5, replace=False) for _ in range(n_cand)]
inner = StratifiedKFold(5, shuffle=True, random_state=1)
outer = StratifiedKFold(5, shuffle=True, random_state=2)
mk = lambda: LogisticRegression(max_iter=1000)

step(f"Flat search: score all {n_cand} candidates on the same folds, report the best")
flat = np.array([cross_val_score(mk(), Xn[:, s], yn, cv=inner).mean()
                 for s in subsets])
print(f"  candidate CV scores: min {flat.min():.4f}  mean {flat.mean():.4f}"
      f"  max {flat.max():.4f}  (std {flat.std():.4f})")
print(f"  best candidate     : #{int(flat.argmax())}")
print(f"  best CV            : {flat.max():.4f}   <- what a flat search reports")
print()
print(f"  Note the mean of all candidates is {flat.mean():.4f} -- close to the")
print(f"  truth. It is only the MAXIMUM that is inflated.")

step("Nested CV: pick inside, score outside")
nested = []
for oi, (otr, ote) in enumerate(outer.split(Xn, yn)):
    # inner loop: choose a candidate using ONLY the outer-training data
    inner_sc = [cross_val_score(mk(), Xn[otr][:, s], yn[otr], cv=inner).mean()
                for s in subsets]
    pick = int(np.argmax(inner_sc))
    m = mk().fit(Xn[otr][:, subsets[pick]], yn[otr])
    sc = m.score(Xn[ote][:, subsets[pick]], yn[ote])
    nested.append(sc)
    print(f"  outer fold {oi}: inner picked candidate #{pick:>2}"
          f" (inner CV {inner_sc[pick]:.4f})  ->  outer score {sc:.4f}")
nested = np.array(nested)
print(f"  nested CV = {nested.mean():.4f}   <- the honest number")

step("The gap")
print(f"  {'flat (chose and reported on the same folds)':>46}  {flat.max():>7.4f}")
print(f"  {'nested (choice kept inside the inner loop)':>46}  {nested.mean():>7.4f}")
print(f"  {'truth (labels are random, classes balanced)':>46}  {0.5:>7.4f}")
print("  " + "-" * 56)
print(f"  {'optimism of the flat search':>46}  {flat.max()-0.5:>+7.4f}")
print(f"  {'optimism of nested CV':>46}  {nested.mean()-0.5:>+7.4f}")
print()
print(f"  The flat search reports {flat.max():.0%} on data containing no signal.")
print(f"  The arithmetic is not wrong -- that really is the best of {n_cand}")
print(f"  scores. But it is the MAXIMUM of {n_cand} noisy numbers, and the")
print(f"  maximum of noise is not zero.")

step("The bias grows with the number of candidates")
print(f"  Given the {n_cand} candidate scores above: if you had tried only n of")
print(f"  them, picked at random, what would your reported best be?")
print(f"  Averaged over 3000 random draws:")
print()
print(f"  {'n tried':>8}  {'E[best CV]':>11}  {'optimism':>9}")
print("  " + "-" * 32)
r2 = np.random.RandomState(7)
prev = None
mono = True
for ncand in [1, 2, 5, 10, 20, 40]:
    draws = [flat[r2.choice(n_cand, ncand, replace=False)].max()
             for _ in range(3000)]
    e = float(np.mean(draws))
    if prev is not None and e < prev - 1e-9:
        mono = False
    prev = e
    print(f"  {ncand:>8}  {e:>11.4f}  {e-0.5:>+9.4f}")
print("  " + "-" * 32)
print(f"  monotonically increasing: {mono}")
print()
print("  Trying one candidate is roughly honest. Trying forty and reporting")
print("  the winner is not. Nothing about the data changed between those")
print("  rows -- only how many chances you gave yourself.")
print()
print("  This is why 'we tried 500 configurations and got 0.94' should raise")
print("  an eyebrow, not confidence.")

step("Being fair to nested CV: unbiased is not the same as exact")
print(f"  Nested CV came out at {nested.mean():.4f} just now -- BELOW the true 0.50.")
print(f"  That is not a failure, it is variance: 5 outer folds of 12 rows each.")
print(f"  The claim is that nested CV has no systematic upward pull, not that")
print(f"  it hits 0.50 on one run. Repeating the whole experiment on 12 fresh")
print(f"  noise datasets:")
print()
print(f"  {'dataset':>8}  {'flat':>8}  {'nested':>8}")
print("  " + "-" * 28)
fl_all, ne_all = [], []
for rep in range(12):
    rr = np.random.RandomState(100 + rep)
    Xr = rr.randn(n_noise, p_noise)
    yr = np.array([0, 1] * (n_noise // 2)); rr.shuffle(yr)
    subs = [rr.choice(p_noise, 5, replace=False) for _ in range(n_cand)]
    f = max(cross_val_score(mk(), Xr[:, s], yr, cv=inner).mean() for s in subs)
    ns = []
    for otr, ote in outer.split(Xr, yr):
        isc = [cross_val_score(mk(), Xr[otr][:, s], yr[otr], cv=inner).mean()
               for s in subs]
        b = subs[int(np.argmax(isc))]
        ns.append(mk().fit(Xr[otr][:, b], yr[otr]).score(Xr[ote][:, b], yr[ote]))
    fl_all.append(f); ne_all.append(np.mean(ns))
    print(f"  {rep:>8}  {f:>8.4f}  {np.mean(ns):>8.4f}")
print("  " + "-" * 28)
fl_all, ne_all = np.array(fl_all), np.array(ne_all)
print(f"  {'mean':>8}  {fl_all.mean():>8.4f}  {ne_all.mean():>8.4f}")
print(f"  {'truth':>8}  {0.5:>8.4f}  {0.5:>8.4f}")
print()
print(f"  flat search   : {fl_all.mean():.4f}"
      f"  ({fl_all.mean()-0.5:+.4f})  above 0.50 on"
      f" {int((fl_all > 0.5).sum())}/12 datasets")
print(f"  nested CV     : {ne_all.mean():.4f}"
      f"  ({ne_all.mean()-0.5:+.4f})  above 0.50 on"
      f" {int((ne_all > 0.5).sum())}/12 datasets")
print()
print("  Averaged over repetitions the flat search sits clearly above chance")
print("  and nested CV sits close to it. That is the difference between a")
print("  biased estimator and a noisy one -- more data fixes noise, but no")
print("  amount of data fixes the bias in a flat search.")

  SELECTION BIAS: FLAT SEARCH vs NESTED CV, ON PURE NOISE
  60 samples, 200 features, all random.
  class balance: [30, 30] -- deliberately exact, so
  the majority-class baseline is 0.50 and cannot be mistaken for skill.
  There is no relationship to learn. Honest accuracy = 0.50.

Flat search: score all 40 candidates on the same folds, report the best
  candidate CV scores: min 0.2833  mean 0.4800  max 0.6167  (std 0.0789)
  best candidate     : #24
  best CV            : 0.6167   <- what a flat search reports

  Note the mean of all candidates is 0.4800 -- close to the
  truth. It is only the MAXIMUM that is inflated.

Nested CV: pick inside, score outside
  outer fold 0: inner picked candidate # 8 (inner CV 0.6644)  ->  outer score 0.4167
  outer fold 1: inner picked candidate #25 (inner CV 0.6467)  ->  outer score 0.3333
  outer fold 2: inner picked candidate # 5 (inner CV 0.6622)  ->  outer score 0.5000
  outer fold 3: inner picked candidate #24 (inner CV 0.7133)  ->  outer score

## 6. What to take away

**A single split is a sample, not a measurement.** It has a standard error, and
on small data that error is large enough to change which model appears to win.

**Cross-validation gives you a distribution.** Use the spread. A difference
smaller than one standard error is not a difference — and because the folds share
training data, that standard error is itself an optimistic floor.

**Match the splitter to the dependence in your data.** Independent rows: k-fold.
Classification: stratify. Repeated measures per subject: group. Ordered in time:
expanding window. Getting this wrong does not produce a noisy answer, it produces
a confident wrong one — the group-vs-plain gap on the EEG data was entirely
manufactured by patients appearing on both sides of the split.

**Selecting is measuring.** Every hyperparameter you compare on a set of folds
spends some of those folds' independence. Tune inside an inner loop, report from
an outer one. The noise experiment is the proof: 200 random features, random
labels, and a flat grid search still reported well above chance.

### Verified in this notebook

- Hand-computed 4-fold scores match `cross_val_score` exactly
- Every recording tested exactly once; fold mean equals pooled accuracy for equal folds
- Stratified folds hold the class ratio far tighter than plain folds
- `GroupKFold` removes the patient overlap that plain k-fold creates, and the accuracy gap that overlap caused
- `TimeSeriesSplit` never places a training row after a test row
- On data with no signal at all, flat search reports well above 0.50 while nested CV does not

### Where this connects

[[data-leakage]] is the other half of this story — cross-validation only protects
you if the folds are honest, and leakage is what makes them dishonest.
[[hyperparameter-search]] is the inner loop of section 5.
[[probability-calibration]] needs out-of-fold predictions, which is the machinery
built here.